# 21 — Storage Cavity Characterization

Consolidates legacy experiments 60 through 64.

1. **Storage spectroscopy** — basic frequency sweep (Exp 60)
2. **Number-splitting spectroscopy** — resolve Fock-dependent frequency peaks (Exp 61)
3. **Storage χ Ramsey** — measure storage-qubit dispersive shift (Exp 62)
4. **Fock-resolved spectroscopy** — spectroscopy conditioned on cavity Fock state (Exp 63)
5. **Fock-resolved T1** — cavity decay per Fock manifold (Exp 64)

## 1. Shared Session Bootstrap

In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import numpy as np
from qualang_tools.units import unit

REPO_ROOT = Path.cwd().resolve()
if not (REPO_ROOT / "qubox").exists():
    REPO_ROOT = Path(r"E:\qubox")
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from qubox.compat.notebook import (
    FockResolvedSpectroscopy,
    FockResolvedT1,
    NumSplittingSpectroscopy,
    StorageChiRamsey,
    StorageSpectroscopy,
    fit_quality_gate,
    load_stage_checkpoint,
    open_notebook_stage,
    preview_or_apply_patch_ops,
    save_stage_checkpoint,
)

REGISTRY_BASE = Path(r"E:\qubox")
SAMPLE_ID = "post_cavity_sample_A"
COOLDOWN_ID = "cd_2025_02_22"
QOP_IP = "10.157.36.68"
CLUSTER_NAME = "Cluster_2"

stage = open_notebook_stage(
    stage_name="21_storage_cavity_characterization",
    registry_base=REGISTRY_BASE,
    sample_id=SAMPLE_ID,
    cooldown_id=COOLDOWN_ID,
    qop_ip=QOP_IP,
    cluster_name=CLUSTER_NAME,
    force_reopen=True,
    close_existing=True,
)
session = stage.session
attr = stage.attr
SESSION_BOOTSTRAP_PATH = stage.bootstrap_path
u = unit(coerce_to_integer=True)

prev_checkpoint = load_stage_checkpoint(
    registry_base=REGISTRY_BASE,
    sample_id=SAMPLE_ID,
    cooldown_id=COOLDOWN_ID,
    stage_name="16_readout_calibration",
)

print(f"Repo root on sys.path: {REPO_ROOT}")
print(f"Shared session bootstrap: {SESSION_BOOTSTRAP_PATH}")
print(f"Stage checkpoint path: {stage.checkpoint_path}")
print(f"QM endpoint: {QOP_IP} ({CLUSTER_NAME})")
if prev_checkpoint is not None:
    print(f"Prior stage 16 status: {prev_checkpoint['status']}")

## 2. Storage Characterization Defaults

In [ ]:
APPLY_STORAGE_CALIBRATION = True

# --- Storage spectroscopy (Exp 60) ---
ST_SPEC_FREQ_START = 6800 * u.MHz
ST_SPEC_FREQ_END = 6810 * u.MHz
ST_SPEC_DF = 100 * u.kHz
ST_SPEC_N_AVG = 10000

# --- Number-splitting spectroscopy (Exp 61) ---
NUM_SPLIT_N_AVG = 20000

# --- Storage χ Ramsey (Exp 62) ---
CHI_RAMSEY_N_AVG = 10000

# --- Fock-resolved spectroscopy (Exp 63) ---
FOCK_SPEC_N_AVG = 10000
FOCK_SPEC_MAX_N = 5

# --- Fock-resolved T1 (Exp 64) ---
FOCK_T1_N_AVG = 10000
FOCK_T1_MAX_N = 5

print("Storage cavity characterization settings:")
print(f"  apply storage calibration: {APPLY_STORAGE_CALIBRATION}")
print(f"  spectroscopy: {ST_SPEC_FREQ_START / 1e9:.3f}–{ST_SPEC_FREQ_END / 1e9:.3f} GHz, df={ST_SPEC_DF / 1e3:.0f} kHz")
print(f"  number-splitting n_avg: {NUM_SPLIT_N_AVG}")
print(f"  Fock-resolved max_n: {FOCK_SPEC_MAX_N}")

## 3. Storage Spectroscopy — Exp 60

Basic storage cavity frequency sweep.

In [ ]:
st_spec = StorageSpectroscopy(session)
st_spec_result = st_spec.run(
    freq_start=ST_SPEC_FREQ_START,
    freq_end=ST_SPEC_FREQ_END,
    df=ST_SPEC_DF,
    n_avg=ST_SPEC_N_AVG,
)
st_spec_analysis = st_spec.analyze(st_spec_result, update_calibration=True)
st_spec.plot(st_spec_analysis)

st_spec_fit_ok = fit_quality_gate(st_spec_analysis, r_squared_min=0.80)

if st_spec_fit_ok:
    st_patch, _, st_apply = preview_or_apply_patch_ops(
        session,
        reason="Storage cavity spectroscopy",
        proposed_patch_ops=st_spec_analysis.metadata.get("proposed_patch_ops", []),
        apply=APPLY_STORAGE_CALIBRATION,
    )
    if st_apply is not None:
        context_snapshot = getattr(session, "context_snapshot", None)
        attr = context_snapshot() if callable(context_snapshot) else getattr(session, "attributes", None)
else:
    print("WARNING: Storage spectroscopy fit quality gate FAILED — skipping apply.")

st_freq_hz = float(st_spec_analysis.metrics.get("f0", np.nan))
print(f"Storage cavity frequency: {st_freq_hz / 1e9:.6f} GHz")

## 4. Number-Splitting Spectroscopy — Exp 61

Resolve Fock-number-dependent qubit frequency peaks.

In [ ]:
num_split = NumSplittingSpectroscopy(session)
num_split_result = num_split.run(n_avg=NUM_SPLIT_N_AVG)
num_split_analysis = num_split.analyze(num_split_result, update_calibration=False)
num_split.plot(num_split_analysis)

print("Number-splitting spectroscopy complete.")
if hasattr(num_split_analysis, "metrics"):
    for k, v in num_split_analysis.metrics.items():
        print(f"  {k}: {v}")

## 5. Storage χ Ramsey — Exp 62

Measure storage-qubit dispersive shift via conditional Ramsey.

In [ ]:
chi_ramsey = StorageChiRamsey(session)
chi_ramsey_result = chi_ramsey.run(n_avg=CHI_RAMSEY_N_AVG)
chi_ramsey_analysis = chi_ramsey.analyze(chi_ramsey_result, update_calibration=False)
chi_ramsey.plot(chi_ramsey_analysis)

storage_chi_hz = float(chi_ramsey_analysis.metrics.get("chi_hz", np.nan))
print(f"Storage-qubit χ: {storage_chi_hz / 1e3:.2f} kHz")

## 6. Fock-Resolved Spectroscopy — Exp 63

Qubit spectroscopy conditioned on storage cavity being in Fock state |n⟩.

In [ ]:
fock_spec = FockResolvedSpectroscopy(session)
fock_spec_result = fock_spec.run(
    max_n=FOCK_SPEC_MAX_N,
    n_avg=FOCK_SPEC_N_AVG,
)
fock_spec_analysis = fock_spec.analyze(fock_spec_result, update_calibration=False)
fock_spec.plot(fock_spec_analysis)

print("Fock-resolved spectroscopy complete.")

## 7. Fock-Resolved T1 — Exp 64

Measure storage cavity decay rate per Fock manifold.

In [ ]:
fock_t1 = FockResolvedT1(session)
fock_t1_result = fock_t1.run(
    max_n=FOCK_T1_MAX_N,
    n_avg=FOCK_T1_N_AVG,
)
fock_t1_analysis = fock_t1.analyze(fock_t1_result, update_calibration=False)
fock_t1.plot(fock_t1_analysis)

print("Fock-resolved T1 complete.")
if hasattr(fock_t1_analysis, "metrics"):
    for k, v in fock_t1_analysis.metrics.items():
        print(f"  {k}: {v}")

## 8. Save Checkpoint

In [ ]:
st_applied = "st_apply" in dir() and st_apply is not None

stage_checkpoint_path = save_stage_checkpoint(
    registry_base=REGISTRY_BASE,
    sample_id=SAMPLE_ID,
    cooldown_id=COOLDOWN_ID,
    stage_name="21_storage_cavity_characterization",
    status="calibrated" if st_applied else "characterized",
    summary="Storage cavity spectroscopy, number-splitting, χ-Ramsey, Fock-resolved spectroscopy and T1.",
    consumed_inputs={
        "st_spec_freq_start": ST_SPEC_FREQ_START,
        "st_spec_freq_end": ST_SPEC_FREQ_END,
        "st_spec_df": ST_SPEC_DF,
        "st_spec_n_avg": ST_SPEC_N_AVG,
        "fock_spec_max_n": FOCK_SPEC_MAX_N,
        "fock_t1_max_n": FOCK_T1_MAX_N,
    },
    persisted_outputs={
        "st_freq_applied": st_applied,
    },
    advisory_outputs={
        "st_freq_hz": st_freq_hz if "st_freq_hz" in dir() else None,
        "storage_chi_hz": storage_chi_hz if "storage_chi_hz" in dir() else None,
    },
    next_stage="22_fock_resolved_experiments",
    notes=[
        "Storage frequency calibration applied if fit passes.",
        "Number-splitting, χ-Ramsey, and Fock-resolved are characterization-only.",
    ],
    metrics={
        "st_freq_hz": st_freq_hz if "st_freq_hz" in dir() else None,
        "storage_chi_hz": storage_chi_hz if "storage_chi_hz" in dir() else None,
    },
)

print(f"Stage checkpoint saved to: {stage_checkpoint_path}")